# TNBC Fusion Clone Gene Dosage Analysis

- Use environment_CNandRNAnotebooks.yml
- Edit the 1st cell below imports to run for either HCC1806 or MDA-MB-231

This notebook tests whether copy number changes in fusion clones, relative
to the additive expectation from their two parental clones, are associated
with matched changes in gene expression (fusion vs mid-parent value, MPV).

Inputs required (see the configuration cell below):

- Per fusion trio, the classified CNV segments table produced by the CNV
  inheritance notebook. This must include the log2_residual column,
  computed as log2 of fusion copy number over the additive expectation of
  the two parents, using continuous ratio-derived copy number values.
- Per fusion trio, the DESeq2 fusion vs mid-parent-value (MPV) results
  table produced by the tagseq analysis notebook.


Outputs produced:

- A per-gene table per fusion trio combining CNV segment data and DESeq2
  results, saved as TSV.
- A Pearson and Spearman correlation scatter plot per fusion trio
  (CNV log2 residual vs DESeq2 log2FC).
- A boxplot per fusion trio comparing DESeq2 log2FC across genes grouped
  by CNV classification (under, neutral, over), with pairwise
  significance brackets.
- An arm level heatmap per fusion trio comparing the bp weighted mean CNV
  log2 residual to the mean DESeq2 log2FC, arm by arm.


## 0. Imports and Configuration

In [ ]:
import os
import pathlib
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import font_manager, rcParams
from matplotlib.colors import TwoSlopeNorm
from matplotlib.cm import ScalarMappable
from scipy.stats import pearsonr, spearmanr, mannwhitneyu, norm, binomtest, fisher_exact
from IPython.display import display

import mygene

pd.set_option("display.max_columns", 50)

In [ ]:
#(Make sure to comment out the set of lines below for the cell line not being analyzed)
# (And edit directory paths below as needed)

CNV_BASE_DIR = "/stor/work/Brock/kennedy/SC_repo/data/GeneDosageAnalysis/CNV_data"
DESEQ_BASE_DIR = "/stor/work/Brock/kennedy/SC_repo/data/GeneDosageAnalysis/deseq_data"

## ---------- Use the lines below for HCC1806 --------------
CELL_LINE_LABEL = "HCC1806"
OUTPUT_DIR = "SC_gene_dosage_effect_1806"
FUSION_TRIOS = {
    "C2C4": {
        "parent_a": "C2_1806_PT",
        "parent_b": "C4_1806_PT",
        "fusion_cnv_name": "C2C4_1806",
        "fusion_deseq_name": "F_C2C4",
        "cnv_segments_path": os.path.join(CNV_BASE_DIR, "C2C4_arm_level_outputs", "02_classified_segments.tsv"),
        "deseq_mpv_path": os.path.join(DESEQ_BASE_DIR, f"{CELL_LINE_LABEL}_F_C2C4_MPV_classification.tsv"),
    },
    "C5C2": {
        "parent_a": "C5_1806_PT",
        "parent_b": "C2_1806_PT",
        "fusion_cnv_name": "C5C2_1806",
        "fusion_deseq_name": "F_C5C2",
        "cnv_segments_path": os.path.join(CNV_BASE_DIR, "C5C2_arm_level_outputs", "02_classified_segments.tsv"),
        "deseq_mpv_path": os.path.join(DESEQ_BASE_DIR, f"{CELL_LINE_LABEL}_F_C5C2_MPV_classification.tsv"),
    },
    "C6C7": {
        "parent_a": "C6_1806_PT",
        "parent_b": "C7_1806_PT",
        "fusion_cnv_name": "C6C7_1806",
        "fusion_deseq_name": "F_C6C7",
        "cnv_segments_path": os.path.join(CNV_BASE_DIR, "C6C7_arm_level_outputs", "02_classified_segments.tsv"),
        "deseq_mpv_path": os.path.join(DESEQ_BASE_DIR, f"{CELL_LINE_LABEL}_F_C6C7_MPV_classification.tsv"),
    },
}

## ---------- Use the lines below for MDA-MB-231 --------------
# CELL_LINE_LABEL = "MDA-MB-231"
# OUTPUT_DIR = "SC_gene_dosage_effect_231"
# FUSION_TRIOS = {
#     "C1C4": {
#         "parent_a": "C1_231_PT",
#         "parent_b": "C4_231_PT",
#         "fusion_cnv_name": "C1C4_231",
#         "fusion_deseq_name": "F_C1C4",
#         "cnv_segments_path": os.path.join(CNV_BASE_DIR, "C1C4_arm_level_outputs", "02_classified_segments.tsv"),
#         "deseq_mpv_path": os.path.join(DESEQ_BASE_DIR, f"{CELL_LINE_LABEL}_F_C1C4_MPV_classification.tsv"),
#     },
#     "C6C8": {
#         "parent_a": "C6_231_PT",
#         "parent_b": "C8_231_PT",
#         "fusion_cnv_name": "C6C8_231",
#         "fusion_deseq_name": "F_C6C8",
#         "cnv_segments_path": os.path.join(CNV_BASE_DIR, "C6C8_arm_level_outputs", "02_classified_segments.tsv"),
#         "deseq_mpv_path": os.path.join(DESEQ_BASE_DIR, f"{CELL_LINE_LABEL}_F_C6C8_MPV_classification.tsv"),
#     },
# }


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
GENE_COORD_CACHE_PATH = os.path.join(OUTPUT_DIR, "gene_coordinates_hg38_cache.tsv")

ALLOWED_CHROMS = [str(i) for i in range(1, 23)] + ["X"]

# If True, segments flagged by any column in CONFIDENCE_FLAG_COLUMNS are
# excluded before computing correlations, boxplots, and arm level means.
EXCLUDE_FLAGGED_SEGMENTS = False
CONFIDENCE_FLAG_COLUMNS = ["low_bin_count", "near_rounding_boundary", "subclone_region_overlap"]

# DESeq2 significance threshold. Used only for descriptive summary counts,
# not for filtering the correlation or boxplot analyses.
PADJ_THRESH_DE = 0.05

# Minimum number of genes with valid DESeq2 data required for an arm to be
# shown in the DE column of the heatmap. Arms below this are shown as
# missing rather than plotted from very few genes.
MIN_GENES_PER_ARM = 3

# mygene lookup settings for gene coordinates (hg38).
MYGENE_SPECIES = "human"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configured fusion trios:", list(FUSION_TRIOS.keys()))
print("Output directory:", OUTPUT_DIR)


## 0b. Figure Style Settings

In [ ]:
# ==============================================================================
#  FIGURE STYLE SETTINGS
# ==============================================================================
# FONT_DIR = pathlib.Path('/stor/work/Brock/kennedy/fonts/arial')
# if FONT_DIR.exists():
#     for _fp in list(FONT_DIR.glob('*.TTF')) + list(FONT_DIR.glob('*.ttf')):
#         font_manager.fontManager.addfont(str(_fp))
#     _avail = [f.name for f in font_manager.fontManager.ttflist]
#     if 'Arial' in _avail:
#         print(f"Arial loaded from {FONT_DIR}")
#     else:
#         print(f"Font files found in {FONT_DIR} but Arial was not registered - using default font.")
# else:
#     print(f"Font dir not found ({FONT_DIR}) - using default font.")

# FONT_FAMILY = "Arial"
FONT_SIZE   = 18
FONT_BOLD   = False
FIG_EXT     = ".png"   # ".svg" or ".png"
FIG_DPI     = 300       # only used when FIG_EXT == ".png"

def apply_style():
    w = "bold" if FONT_BOLD else "normal"
    rcParams.update({
        # "font.family":           FONT_FAMILY,
        "font.size":             FONT_SIZE,
        "font.weight":           w,
        "axes.titlesize":        FONT_SIZE + 1,
        "axes.titleweight":      w,
        "axes.labelsize":        FONT_SIZE,
        "axes.labelweight":      w,
        "xtick.labelsize":       FONT_SIZE - 1,
        "ytick.labelsize":       FONT_SIZE - 1,
        "legend.fontsize":       FONT_SIZE - 1,
        "legend.title_fontsize": FONT_SIZE,
        "figure.titlesize":      FONT_SIZE + 2,
        "figure.titleweight":    w,
    })

apply_style()

def save_fig(fig, name, tight=True):
    path = os.path.join(OUTPUT_DIR, f"{name}{FIG_EXT}")
    kw = {"bbox_inches": "tight"} if tight else {}
    if FIG_EXT.lower() == ".png":
        kw["dpi"] = FIG_DPI
    fig.savefig(path, **kw)
    print(f"Saved: {path}")
    return path

print(f"Style applied  |  FIG_EXT={FIG_EXT}  |  FONT_SIZE={FONT_SIZE}  |  FONT_BOLD={FONT_BOLD}")


## 1. Load CNV Segment and DESeq2 Data

In [ ]:
def load_cnv_segments(path, confidence_flag_columns, exclude_flagged=True):
    """
    Load one fusion trio's classified segments TSV.
    Optionally excludes segments flagged by any column in
    confidence_flag_columns. Returns the cleaned dataframe and a small
    report dictionary.
    """
    df = pd.read_csv(path, sep="\t")
    df["Chromosome"] = df["Chromosome"].astype(str)

    n_total = len(df)
    flag_cols_present = [c for c in confidence_flag_columns if c in df.columns]

    if exclude_flagged and flag_cols_present:
        flagged_mask = df[flag_cols_present].any(axis=1)
    else:
        flagged_mask = pd.Series(False, index=df.index)

    df_clean = df.loc[~flagged_mask].copy().reset_index(drop=True)

    report = {
        "n_segments_total": n_total,
        "n_segments_flagged": int(flagged_mask.sum()),
        "n_segments_kept": len(df_clean),
    }
    return df_clean, report


def load_deseq_mpv(path):
    """
    Load one fusion trio's DESeq2 fusion vs MPV classification TSV.
    The gene symbol is expected in the first, unnamed column, which is
    used as the index.
    """
    df = pd.read_csv(path, sep="\t", index_col=0)
    df.index = df.index.astype(str)
    df = df[~df.index.duplicated(keep="first")]

    keep_cols = [c for c in ["f_mpv_log2FoldChange", "f_mpv_padj", "category"] if c in df.columns]
    return df[keep_cols].copy()


In [ ]:
cnv_data = {}
cnv_reports = {}
deseq_data = {}

for trio_name, cfg in FUSION_TRIOS.items():
    seg_df, report = load_cnv_segments(
        cfg["cnv_segments_path"], CONFIDENCE_FLAG_COLUMNS, EXCLUDE_FLAGGED_SEGMENTS
    )
    cnv_data[trio_name] = seg_df
    cnv_reports[trio_name] = report

    de_df = load_deseq_mpv(cfg["deseq_mpv_path"])
    deseq_data[trio_name] = de_df

    print(
        f"{trio_name} - segments kept: {report['n_segments_kept']} of {report['n_segments_total']} "
        f"({report['n_segments_flagged']} flagged and excluded) - DE genes loaded: {len(de_df)}"
    )


## 2. Gene Coordinate Annotation (mygene, hg38)

Gene symbols are mapped to hg38 genomic coordinates using mygene, the same
package already used in the tagseq notebook for Entrez ID mapping.
mygene's genomic_pos field is based on the hg38 assembly for human genes,
matching the hg38 centromere source used in the CNV inheritance notebook.

Results are cached to a local TSV so repeated runs of this notebook do not
re-query mygene for genes already looked up.


In [ ]:
def get_gene_coordinates(gene_list, cache_path, species="human", allowed_chroms=None):
    """
    Query mygene for hg38 genomic coordinates of each gene symbol in
    gene_list. Uses a local cache file if present so repeated runs do not
    re-query genes that were already looked up. Returns a dataframe
    indexed by gene symbol with chrom, start, end, and mid columns.
    """
    gene_list = sorted(set(gene_list))

    if os.path.exists(cache_path):
        cached = pd.read_csv(cache_path, sep="\t", index_col=0)
        cached.index = cached.index.astype(str)
        missing = [g for g in gene_list if g not in cached.index]
        if not missing:
            print("Loaded gene coordinates from cache:", cache_path)
            return cached.loc[cached.index.isin(gene_list)]
        print("Cache found but missing", len(missing), "genes, querying those now.")
        query_list = missing
    else:
        cached = None
        query_list = gene_list

    mg = mygene.MyGeneInfo()
    # hits = mg.querymany(
    #     query_list,
    #     scopes="symbol",
    #     fields="genomic_pos",
    #     species=species,
    #     verbose=False,
    # )
    hits = mg.querymany(
        query_list,
        scopes="symbol,alias",
        fields="genomic_pos",
        species=species,
        verbose=False,
    )

    hits_by_query = defaultdict(list)
    for hit in hits:
        hits_by_query[hit.get("query")].append(hit)

    rows = []
    n_unmapped = 0
    n_ambiguous = 0

    for gene, gene_hits in hits_by_query.items():
        valid_hits = [h for h in gene_hits if not h.get("notfound") and "genomic_pos" in h]
        if not valid_hits:
            n_unmapped += 1
            continue
        if len(valid_hits) > 1:
            n_ambiguous += 1

        best_hit = sorted(valid_hits, key=lambda h: h.get("_score", 0), reverse=True)[0]
        pos = best_hit["genomic_pos"]
        if isinstance(pos, list):
            pos = sorted(pos, key=lambda p: p.get("end", 0) - p.get("start", 0), reverse=True)[0]

        rows.append({
            "gene": gene,
            "chrom": str(pos.get("chr")),
            "start": pos.get("start"),
            "end": pos.get("end"),
        })

    if rows:
        new_df = pd.DataFrame(rows).drop_duplicates(subset="gene", keep="first").set_index("gene")
    else:
        new_df = pd.DataFrame(columns=["chrom", "start", "end"])

    if allowed_chroms is not None and len(new_df):
        new_df = new_df[new_df["chrom"].isin(allowed_chroms)]

    if len(new_df):
        new_df["mid"] = (new_df["start"] + new_df["end"]) / 2.0

    if cached is not None:
        combined = pd.concat([cached, new_df])
        combined = combined[~combined.index.duplicated(keep="first")]
    else:
        combined = new_df

    combined.to_csv(cache_path, sep="\t")

    print("Gene coordinate lookup complete.")
    print(f"  Queried: {len(query_list)}")
    print(f"  Unmapped: {n_unmapped}")
    print(f"  Ambiguous (multiple hits, kept highest scoring): {n_ambiguous}")
    print(f"  Total coordinates now cached: {len(combined)}")

    return combined.loc[combined.index.isin(gene_list)]


In [ ]:
all_genes = sorted(set().union(*[set(df.index) for df in deseq_data.values()]))
print("Total unique genes with DESeq2 results across all trios:", len(all_genes))

gene_coords = get_gene_coordinates(
    all_genes, GENE_COORD_CACHE_PATH, species=MYGENE_SPECIES, allowed_chroms=ALLOWED_CHROMS
)
gene_coords.head()


## 3. Map Genes to CNV Segments

Each gene is assigned to the CNV segment whose start to end range contains
the gene's coordinate midpoint, on the matching chromosome, using that
fusion trio's own segment set. Genes on a chromosome with no segment data,
or that fall outside every segment, are excluded and counted below.


In [ ]:
def map_genes_to_segments(seg_df, gene_coords_df):
    """
    Map each gene, using its coordinate midpoint, to the CNV segment whose
    Start to End range contains it, on the same chromosome. Returns a
    dataframe indexed by gene with the matching segment's columns
    attached, restricted to genes that fall inside exactly one segment.
    """
    seg_cols = [
        "Arm", "log2_residual", "residual", "classification",
        "is_informative", "low_bin_count", "near_rounding_boundary",
        "subclone_region_overlap", "segment_bp", "Segment_ID", "Start", "End",
    ]
    seg_cols = [c for c in seg_cols if c in seg_df.columns]

    # closed="left" treats segments as half-open [Start, End), the standard
    # genomic convention. This matters because consecutive segments touch
    # (one segment's End equals the next segment's Start): with closed="both"
    # a gene sitting exactly on that boundary matches both segments at once,
    # which pandas returns as a slice rather than a single position, and
    # that slice would silently pull a whole sub-dataframe into a spot that
    # expects one row. closed="left" resolves the boundary to the segment
    # that starts there, with no ambiguity.
    interval_by_chrom = {}
    for chrom, sub in seg_df.groupby("Chromosome"):
        sub_sorted = sub.sort_values("Start").reset_index(drop=True)
        idx = pd.IntervalIndex.from_arrays(sub_sorted["Start"], sub_sorted["End"], closed="left")
        interval_by_chrom[chrom] = (idx, sub_sorted)

    rows = []
    n_no_chrom = 0
    n_no_segment = 0

    for gene, row in gene_coords_df.iterrows():
        chrom = row["chrom"]
        mid = row["mid"]

        if chrom not in interval_by_chrom:
            n_no_chrom += 1
            continue

        idx, sub_sorted = interval_by_chrom[chrom]
        try:
            pos = idx.get_loc(mid)
        except KeyError:
            n_no_segment += 1
            continue

        if isinstance(pos, slice):
            pos = range(pos.start, pos.stop)[0] if pos.stop > pos.start else None
            if pos is None:
                n_no_segment += 1
                continue
        elif isinstance(pos, (np.ndarray, list)):
            if len(pos) == 0:
                n_no_segment += 1
                continue
            pos = pos[0]

        seg_row = sub_sorted.iloc[pos]
        out = {"gene": gene, "chrom": chrom, "gene_mid": mid}
        for c in seg_cols:
            out[c] = seg_row[c]
        rows.append(out)

    if rows:
        result = pd.DataFrame(rows).set_index("gene")
    else:
        result = pd.DataFrame(columns=["chrom", "gene_mid"] + seg_cols)

    print(f"  Genes with coordinates: {len(gene_coords_df)}")
    print(f"  Mapped to a segment: {len(result)}")
    print(f"  Chromosome not present in segment data: {n_no_chrom}")
    print(f"  No overlapping segment found: {n_no_segment}")

    return result


In [ ]:
gene_level_data = {}

for trio_name, cfg in FUSION_TRIOS.items():
    print(trio_name)
    seg_df = cnv_data[trio_name]
    de_df = deseq_data[trio_name]

    trio_genes = gene_coords.loc[gene_coords.index.isin(de_df.index)]
    mapped = map_genes_to_segments(seg_df, trio_genes)

    merged = mapped.join(de_df, how="inner")
    gene_level_data[trio_name] = merged

    out_path = os.path.join(OUTPUT_DIR, f"{trio_name}_gene_level_dosage_table.tsv")
    merged.to_csv(out_path, sep="\t")

    print(f"  Final gene level rows (segment and DE data both present): {len(merged)}")
    print(f"  Saved to: {out_path}")
    print()


## 4. Continuous Correlation: Pearson and Spearman

For each fusion trio, this correlates the CNV log2 residual of the
segment each gene falls in against that gene's DESeq2 log2FC (fusion vs
MPV). This is the direct, threshold free test of the dosage hypothesis,
matching the continuous correlation step used in the earlier bulk WGD
dosage analysis, now applied per matched fusion trio.


In [ ]:
CLASS_COLORS = {"match": "#808080", "over": "#B03A2E", "under": "#2E4E9E"}
CLASS_DISPLAY = {"match": "Neutral", "over": "Over", "under": "Under"}
CLASS_ORDER = ["under", "match", "over"]


def compute_correlations(df, x_col="log2_residual", y_col="f_mpv_log2FoldChange"):
    sub = df[[x_col, y_col]].dropna()
    if len(sub) < 3:
        return {
            "n_genes": len(sub), "pearson_r": np.nan, "pearson_p": np.nan,
            "spearman_r": np.nan, "spearman_p": np.nan,
        }
    pear_r, pear_p = pearsonr(sub[x_col], sub[y_col])
    spear_r, spear_p = spearmanr(sub[x_col], sub[y_col])
    return {
        "n_genes": len(sub),
        "pearson_r": pear_r,
        "pearson_p": pear_p,
        "spearman_r": spear_r,
        "spearman_p": spear_p,
    }


def plot_dosage_correlation(merged_df, trio_name, ax, x_col="log2_residual", y_col="f_mpv_log2FoldChange"):
    sub = merged_df[[x_col, y_col, "classification"]].dropna(subset=[x_col, y_col])

    for cls in CLASS_ORDER:
        cls_sub = sub[sub["classification"] == cls]
        ax.scatter(
            cls_sub[x_col], cls_sub[y_col], s=10, alpha=0.5,
            color=CLASS_COLORS[cls], label=CLASS_DISPLAY[cls], linewidths=0,
        )

    stats_dict = compute_correlations(sub, x_col, y_col)

    if len(sub) > 1:
        slope, intercept = np.polyfit(sub[x_col], sub[y_col], 1)
        x_line = np.linspace(sub[x_col].min(), sub[x_col].max(), 100)
        ax.plot(x_line, slope * x_line + intercept, color="black", linewidth=1.5)

    ax.axhline(0, color="grey", linewidth=0.5, linestyle="--")
    ax.axvline(0, color="grey", linewidth=0.5, linestyle="--")
    ax.set_xlabel(r"log$_2$CNV residual""\n(fusion vs additive expectation)")
    ax.set_ylabel(r"DESeq2 log$_2$FC""\n(fusion vs mid-parent value)")
    ax.set_title(f"Fusion {trio_name}")

    text = (
        f"n = {stats_dict['n_genes']}\n"
        f"Pearson r = {stats_dict['pearson_r']:.3f}, p = {stats_dict['pearson_p']:.2e}\n"
        f"Spearman rho = {stats_dict['spearman_r']:.3f}, p = {stats_dict['spearman_p']:.2e}"
    )
    ax.text(0.5, -0.32, text, transform=ax.transAxes, va="top", ha="center", fontsize=FONT_SIZE - 2)

    return stats_dict


In [ ]:
fig, axes = plt.subplots(1, len(FUSION_TRIOS), figsize=(6 * len(FUSION_TRIOS), 5), constrained_layout=True)
if len(FUSION_TRIOS) == 1:
    axes = [axes]

correlation_summary = []
for ax, trio_name in zip(axes, FUSION_TRIOS.keys()):
    stats_dict = plot_dosage_correlation(gene_level_data[trio_name], trio_name, ax)
    stats_dict["trio"] = trio_name
    correlation_summary.append(stats_dict)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=len(labels), frameon=False,
           bbox_to_anchor=(0.5, -0.18), markerscale=3)
           
save_fig(fig, "dosage_correlation_scatter")
plt.show()

correlation_summary_df = pd.DataFrame(correlation_summary)[
    ["trio", "n_genes", "pearson_r", "pearson_p", "spearman_r", "spearman_p"]
]
correlation_summary_df.to_csv(os.path.join(OUTPUT_DIR, "correlation_summary.tsv"), sep="\t", index=False)
correlation_summary_df


## 5. Boxplot by CNV Classification (Under, Neutral, Over)

Genes are grouped by the classification of the CNV segment they fall in
(under, match, over), using the exact integer classification already
computed in the CNV notebook, not a threshold on log2_residual. The
match group is labeled Neutral in the plot. Styling matches the Part 1
plot_dosage_boxplots_signonly function: no whiskers, a percentile
tightened y-axis (10th to 97th percentile of the data, not the full
range) so a handful of extreme genes cannot compress the rest of the
plot, median lines colored to match their own box rather than the
matplotlib default, and a jittered strip plot of individual genes behind
the boxes.

Two complementary statistical tests are run per fusion trio:

- Pairwise two sided Mann Whitney U tests (under vs neutral, neutral vs
  over, under vs over), each with a Bonferroni correction of x3 for the
  three comparisons within that trio, matching the Part 1 convention.
  Drawn on the plot as tiered brackets, and saved in full (raw and
  corrected p-values, group sizes, medians) to a TSV.
- A Jonckheere-Terpstra test, which tests the specific ordered
  alternative the dosage hypothesis actually predicts (under < neutral <
  over) directly, rather than as three separate pairwise comparisons.
  Run one sided for an increasing trend, since that is the hypothesis
  being tested here, and reported in a separate TSV rather than drawn on
  the plot.

As discussed, both tests treat each gene as an independent observation,
which is not strictly true since genes on the same CNV segment are
physically linked. With several thousand genes per group this makes the
p-values from either test anti-conservative (too easily significant),
so the more trustworthy evidence for a real dosage effect is a
consistent direction and effect size across all three fusion trios, not
the p-value from any single trio on its own.


In [ ]:
EDGE = {"under": "#3B6CC5", "match": "#888888", "over": "#C43E3E"}
FACE = {"under": "#DBEAFE", "match": "#F3F4F6", "over": "#FEE2E2"}
GROUP_KEYS = ["under", "match", "over"]
GROUP_LABS = ["Under-duplicated", "Neutral (exact match)", "Over-duplicated"]

GW = 0.65    # box slot width within a trio group
GAP = 0.95   # gap between trio groups
COMPARISONS = [(0, 1, 0), (1, 2, 0), (0, 2, 1)]   # (i, j, bracket tier)


def significance_label(p):
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def draw_bracket(ax, x1, x2, y_base, h, label, color="#333333", lw=0.8, fontsize=FONT_SIZE - 4):
    ax.plot([x1, x1, x2, x2], [y_base, y_base + h, y_base + h, y_base], lw=lw, color=color, clip_on=False)
    ax.text((x1 + x2) / 2, y_base + h, label, ha="center", va="bottom", fontsize=fontsize, color=color, clip_on=False)


def jonckheere_terpstra_test(groups, alternative="increasing"):
    """
    Jonckheere-Terpstra test for a monotone trend across ordered groups.
    groups must be a list of arrays given in the intended increasing
    order (here: under, match, over). Uses the standard normal
    approximation for the p-value, which is appropriate given the group
    sizes here (each group has at least tens, usually thousands, of
    genes). alternative is "increasing", "decreasing", or "two-sided".
    """
    k = len(groups)
    n = [len(g) for g in groups]
    N = sum(n)

    if any(ni == 0 for ni in n):
        return {"JT": np.nan, "z": np.nan, "p_value": np.nan, "log10_p_value": np.nan, "n_total": N}

    JT = 0.0
    for i in range(k):
        for j in range(i + 1, k):
            u_ji = mannwhitneyu(groups[j], groups[i], alternative="two-sided").statistic
            JT += u_ji

    mean_JT = (N ** 2 - sum(ni ** 2 for ni in n)) / 4.0
    var_JT = (N ** 2 * (2 * N + 3) - sum(ni ** 2 * (2 * ni + 3) for ni in n)) / 72.0

    if var_JT <= 0:
        return {"JT": JT, "z": np.nan, "p_value": np.nan, "log10_p_value": np.nan, "n_total": N}

    z = (JT - mean_JT) / np.sqrt(var_JT)

    # norm.sf is used instead of 1 - norm.cdf because for z beyond about 8,
    # cdf(z) itself rounds to exactly 1.0 in float64, which would make
    # 1 - cdf(z) collapse to 0.0 even though the true p-value is not
    # actually zero. norm.sf computes the same quantity directly and
    # stays accurate much further out.
    #
    # For z beyond about 37, even norm.sf underflows to exactly 0.0,
    # because the true p-value is smaller than the smallest number a
    # float64 can represent at all (this is a real hardware limit, not a
    # bug). log10_p_value is computed via logsf, which stays accurate
    # arbitrarily far out, so there is still a meaningful number to look
    # at even when p_value itself has hit the float64 floor.
    if alternative == "increasing":
        p = norm.sf(z)
        log10_p = norm.logsf(z) / np.log(10)
    elif alternative == "decreasing":
        p = norm.cdf(z)
        log10_p = norm.logcdf(z) / np.log(10)
    else:
        p = 2.0 * norm.sf(abs(z))
        log10_p = np.log(2.0) / np.log(10) + norm.logsf(abs(z)) / np.log(10)

    return {"JT": JT, "z": z, "p_value": p, "log10_p_value": log10_p, "n_total": N}


def plot_dosage_boxplot_all_trios(gene_level_data, fusion_trios, y_col="f_mpv_log2FoldChange",
                                  show_stripplot=True, strip_n=400, strip_alpha=0.5,
                                  strip_size=1.5, seed=42):
    """
    One panel, x-axis grouped by fusion trio, three boxes per trio (under,
    match, over). Styled to match the Part 1 plot_dosage_boxplots_signonly
    function. Returns the figure, axes, the pairwise Mann-Whitney U stats
    table, and the Jonckheere-Terpstra trend test table.
    """
    trio_names = list(fusion_trios.keys())

    all_data = []
    all_pos = []
    tick_pos = []
    tick_labs = []
    group_info = []
    strip_buffer = []

    for ti, trio_name in enumerate(trio_names):
        base = ti * (2 * GW + GAP)
        trio_positions = [base + i * GW for i in range(3)]

        df = gene_level_data[trio_name][["classification", y_col]].dropna()

        group_vals = []
        for bi, cls in enumerate(GROUP_KEYS):
            vals = df.loc[df["classification"] == cls, y_col].values
            pos = trio_positions[bi]
            all_data.append(vals if len(vals) > 0 else np.array([np.nan]))
            group_vals.append(vals)
            all_pos.append(pos)
            if len(vals) > 0:
                strip_buffer.append((pos, vals, cls))

        tick_pos.append(base + GW)
        tick_labs.append(f"Fusion {trio_name}")
        group_info.append({"positions": trio_positions, "vals": group_vals, "trio": trio_name})

    fig, ax = plt.subplots(figsize=(2.6 * len(trio_names) + 1, 5))
    fig.subplots_adjust(bottom=0.2)

    if show_stripplot:
        rng = np.random.default_rng(seed=seed)
        for xcen, vals, cls in strip_buffer:
            samp = rng.choice(vals, size=min(strip_n, len(vals)), replace=False) if strip_n else vals
            jitter = rng.uniform(-GW * 0.30, GW * 0.30, size=len(samp))
            ax.scatter(xcen + jitter, samp, s=strip_size, color="#555555", alpha=strip_alpha,
                       linewidths=0, zorder=1, rasterized=True)

    bp = ax.boxplot(
        all_data, positions=all_pos, widths=GW * 0.72, patch_artist=True, showfliers=False,
        medianprops=dict(linewidth=2.0, zorder=4),
        whiskerprops=dict(linewidth=0, zorder=3),
        capprops=dict(linewidth=0, zorder=3),
        boxprops=dict(linewidth=0.9, zorder=2),
    )
    for i, box in enumerate(bp["boxes"]):
        k = GROUP_KEYS[i % 3]
        box.set_facecolor(FACE[k])
        box.set_edgecolor(EDGE[k])
        box.set_alpha(0.85 if show_stripplot else 1.0)
    for i, med in enumerate(bp["medians"]):
        med.set_color(EDGE[GROUP_KEYS[i % 3]])

    ax.axhline(0, color="#444444", linewidth=0.7, linestyle="--", alpha=0.5, zorder=0)
    for ti in range(1, len(trio_names)):
        sep = ti * (2 * GW + GAP) - GAP / 2
        ax.axvline(sep, color="#DDDDDD", linewidth=0.6)

    bracket_specs = []
    pairwise_stats = []
    trend_stats = []

    for gi in group_info:
        pos = gi["positions"]
        dat = gi["vals"]
        trio_name = gi["trio"]

        q3_list = [float(np.percentile(d, 75)) for d in dat if len(d) >= 2]
        base_y = max(q3_list) if q3_list else 0.0

        for (i, j, tier) in COMPARISONS:
            d_i, d_j = dat[i], dat[j]
            if len(d_i) < 2 or len(d_j) < 2:
                continue
            u_stat, p_mw = mannwhitneyu(d_i, d_j, alternative="two-sided")
            p_bonf = min(p_mw * 3, 1.0)
            label = significance_label(p_bonf)

            bracket_specs.append({"x1": pos[i], "x2": pos[j], "base_y": base_y, "tier": tier, "label": label})
            pairwise_stats.append({
                "trio": trio_name,
                "group_a": GROUP_KEYS[i], "n_a": len(d_i), "median_a": round(float(np.median(d_i)), 5),
                "group_b": GROUP_KEYS[j], "n_b": len(d_j), "median_b": round(float(np.median(d_j)), 5),
                "U_statistic": round(float(u_stat), 1),
                "p_raw": p_mw, "p_bonferroni_x3": p_bonf, "significance": label,
            })

        if all(len(d) >= 2 for d in dat):
            jt_result = jonckheere_terpstra_test(dat, alternative="increasing")
            trend_stats.append({
                "trio": trio_name,
                "n_under": len(dat[0]), "n_match": len(dat[1]), "n_over": len(dat[2]),
                "JT_statistic": jt_result["JT"], "z": jt_result["z"],
                "p_value_one_sided_increasing": jt_result["p_value"],
                "log10_p_value_one_sided_increasing": jt_result["log10_p_value"],
            })

    all_group_arrays = [d for gi in group_info for d in gi["vals"] if len(d) >= 2]
    all_vals = np.concatenate(all_group_arrays)

    # Box extent (this must never be clipped, no matter what).
    q1s = [float(np.percentile(d, 25)) for d in all_group_arrays]
    q3s = [float(np.percentile(d, 75)) for d in all_group_arrays]
    box_lo = min(q1s)
    box_hi = max(q3s)

    # Two candidate ranges beyond the boxes:
    #  (a) 10th/97th percentile of the pooled data (can be pulled far by a
    #      broadly skewed distribution)
    #  (b) each group's own 1.5xIQR Tukey whisker extent, the same rule
    #      that already defines which points showfliers=False is hiding
    # Using whichever candidate is tighter on each side means neither a
    # skewed tail nor one noisy group can single-handedly stretch the
    # axis and push the boxes toward one edge of the plot.
    pct_lo = float(np.percentile(all_vals, 10))
    pct_hi = float(np.percentile(all_vals, 97))

    whisk_los, whisk_his = [], []
    for d in all_group_arrays:
        q1, q3 = np.percentile(d, [25, 75])
        iqr = q3 - q1
        whisk_los.append(max(float(d.min()), q1 - 1.5 * iqr))
        whisk_his.append(min(float(d.max()), q3 + 1.5 * iqr))
    whisk_lo = min(whisk_los)
    whisk_hi = max(whisk_his)

    y_lo_tight = min(box_lo, max(pct_lo, whisk_lo))
    y_hi_tight = max(box_hi, min(pct_hi, whisk_hi))

    # Small fixed margin so the boxes never sit flush against the axis edge.
    margin = 0.08 * (box_hi - box_lo) if box_hi > box_lo else 0.1
    y_lo_tight -= margin
    y_hi_tight += margin

    tight_range = y_hi_tight - y_lo_tight

    pad_y = tight_range * 0.04
    step_y = tight_range * 0.065
    bkt_h = tight_range * 0.012

    max_y_needed = y_hi_tight
    for spec in bracket_specs:
        y_top = spec["base_y"] + pad_y + spec["tier"] * step_y + bkt_h + tight_range * 0.025
        max_y_needed = max(max_y_needed, y_top)

    ax.set_ylim(y_lo_tight, max_y_needed + tight_range * 0.05)

    for spec in bracket_specs:
        y_b = spec["base_y"] + pad_y + spec["tier"] * step_y
        draw_bracket(ax, spec["x1"], spec["x2"], y_b, bkt_h, spec["label"])

    ax.set_xticks(tick_pos)
    ax.set_xticklabels(tick_labs)
    ax.set_xlim(-GW * 0.6, all_pos[-1] + GW * 0.6)
    ax.set_ylabel( r"DESeq2 log$_2$FC" "\n" r"(fusion vs mid-parent value)")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    fig.legend(
        handles=[mpatches.Patch(facecolor=FACE[k], edgecolor=EDGE[k], linewidth=0.9, label=lab)
                 for k, lab in zip(GROUP_KEYS, GROUP_LABS)],
        loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.0),
        fontsize=FONT_SIZE-2
    )

    return fig, ax, pd.DataFrame(pairwise_stats), pd.DataFrame(trend_stats)

In [ ]:
fig, ax, pairwise_stats_df, trend_stats_df = plot_dosage_boxplot_all_trios(
    gene_level_data, FUSION_TRIOS,
    show_stripplot=True, strip_n=400, strip_alpha=0.5, strip_size=1.5, seed=42,
)

save_fig(fig, "dosage_boxplot_by_classification")
plt.show()

pairwise_stats_df.to_csv(os.path.join(OUTPUT_DIR, "boxplot_pairwise_mannwhitney_stats.tsv"), sep="\t", index=False)
trend_stats_df.to_csv(os.path.join(OUTPUT_DIR, "boxplot_jonckheere_terpstra_trend_stats.tsv"), sep="\t", index=False)

print("Pairwise Mann-Whitney U tests (Bonferroni x3 within each trio):")
display(pairwise_stats_df)
print()
print("Jonckheere-Terpstra trend test (one sided, increasing under < neutral < over):")
display(trend_stats_df)


## 6. Arm Level Heatmap: CNV Residual vs Expression

For each arm, the CNV side is the base pair weighted mean log2_residual
across that arm's segments, using the same weighting convention as the
arm level rollup in the CNV notebook, computed from all segments on that
arm regardless of whether every gene there has DESeq2 data. The
expression side is the plain mean DESeq2 log2FC across genes mapped to
that arm. Arms with fewer than MIN_GENES_PER_ARM genes on the expression
side are shown as missing (grey) rather than plotted from very few genes.

Styled to match the Part 1 plot_arm_heatmap function: one horizontal
heatmap, chromosome arms in genomic order along the x-axis, with two
rows per fusion trio (CNV log2 residual, DE mean log2FC) stacked
vertically rather than as separate side by side plots. All trios and
both value types share one color scale, since both are now expressed as
log2 ratios and are intended to be directly comparable in magnitude.
Unlike Part 1's version, there is only one cell line here so no side by
side panel is needed, and no significance markers are drawn on the DNA
rows.


In [ ]:
def sort_arms(arms):
    """
    Sort chromosome arm labels such as 1p, 1q, 2p, 10q, Xp in genomic
    order.
    """
    def arm_key(arm):
        arm = str(arm)
        if arm.endswith("p"):
            chrom, side = arm[:-1], 0
        elif arm.endswith("q"):
            chrom, side = arm[:-1], 1
        else:
            chrom, side = arm, 2

        if chrom == "X":
            chrom_num = 23
        elif chrom == "Y":
            chrom_num = 24
        else:
            try:
                chrom_num = int(chrom)
            except ValueError:
                chrom_num = 99
        return (chrom_num, side)

    return sorted(set(arms), key=arm_key)


def compute_arm_level_cnv(seg_df):
    rows = []
    for arm, g in seg_df.groupby("Arm"):
        g_valid = g.dropna(subset=["log2_residual"])
        total_bp = g_valid["segment_bp"].sum() if "segment_bp" in g_valid.columns else 0
        if len(g_valid) == 0 or total_bp == 0:
            weighted_mean = np.nan
        else:
            weighted_mean = np.average(g_valid["log2_residual"], weights=g_valid["segment_bp"])
        rows.append({"Arm": arm, "cnv_log2_residual": weighted_mean, "n_segments": len(g)})
    return pd.DataFrame(rows).set_index("Arm")


def compute_arm_level_de(merged_df, min_genes_per_arm, y_col="f_mpv_log2FoldChange"):
    rows = []
    for arm, g in merged_df.dropna(subset=[y_col]).groupby("Arm"):
        n = len(g)
        mean_lfc = g[y_col].mean() if n >= min_genes_per_arm else np.nan
        rows.append({"Arm": arm, "de_mean_log2FC": mean_lfc, "n_genes": n})
    return pd.DataFrame(rows).set_index("Arm")


In [ ]:
arm_level_by_trio = {}

for trio_name in FUSION_TRIOS.keys():
    cnv_arm = compute_arm_level_cnv(cnv_data[trio_name])
    de_arm = compute_arm_level_de(gene_level_data[trio_name], MIN_GENES_PER_ARM)
    combined = cnv_arm.join(de_arm, how="outer")
    arm_level_by_trio[trio_name] = combined

    out_path = os.path.join(OUTPUT_DIR, f"{trio_name}_arm_level_dosage_summary.tsv")
    combined.to_csv(out_path, sep="\t")
    print(
        f"{trio_name} - arms with CNV data: {combined['cnv_log2_residual'].notna().sum()} "
        f"- arms with DE data (n >= {MIN_GENES_PER_ARM} genes): {combined['de_mean_log2FC'].notna().sum()}"
    )


In [ ]:
def plot_combined_arm_heatmap(arm_level_by_trio, fusion_trios, vmax=None):
    """
    One heatmap, chromosome arms in genomic order along the x-axis, two
    rows per fusion trio (CNV log2 residual, DE mean log2FC) stacked
    vertically. Styled to match the Part 1 plot_arm_heatmap function
    (TwoSlopeNorm centered at 0, white chromosome divider lines, white
    row divider lines between trio blocks), simplified to one cell line
    and no significance markers.
    """
    trio_names = list(fusion_trios.keys())

    all_arms_set = set()
    for trio_name in trio_names:
        all_arms_set.update(arm_level_by_trio[trio_name].index)
    arm_order = sort_arms(all_arms_set)

    row_labels = []
    matrix = []
    for trio_name in trio_names:
        df = arm_level_by_trio[trio_name].reindex(arm_order)
        matrix.append(df["cnv_log2_residual"].values)
        row_labels.append(f"Fusion {trio_name}\n"r"log$_2$ CNV residual")
        matrix.append(df["de_mean_log2FC"].values)
        row_labels.append(f"Fusion {trio_name}\n"r"DE mean log$_2$FC")

    mat = np.array(matrix, dtype=float)

    if vmax is None:
        finite_vals = mat[np.isfinite(mat)]
        vmax = float(np.percentile(np.abs(finite_vals), 99)) if len(finite_vals) else 1.0
        if vmax == 0:
            vmax = 1.0

    cmap = plt.get_cmap("RdBu_r")
    plot_norm = TwoSlopeNorm(vcenter=0, vmin=-vmax, vmax=vmax)

    fig, ax = plt.subplots(figsize=(20, 0.55 * len(row_labels) + 1.5))

    ax.imshow(mat, aspect="auto", cmap=cmap, norm=plot_norm, interpolation="nearest")

    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            if not np.isfinite(mat[i, j]):
                ax.add_patch(mpatches.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=True, color="lightgrey"))

    def _chrom_label(chrom, start, end):
        """end is exclusive; label includes arm letter if only one arm present."""
        if end - start == 1:
            return arm_order[start]  # e.g. "13q" instead of just "13"
        return chrom

    xtick_pos, xtick_lab = [], []
    prev_chrom = None
    chrom_start = None
    for xi, arm in enumerate(arm_order):
        chrom = arm[:-1]
        if chrom != prev_chrom:
            if prev_chrom is not None:
                ax.axvline(xi - 0.5, color="white", linewidth=1.2, zorder=3)
                xtick_pos.append((chrom_start + xi - 1) / 2)
                xtick_lab.append(_chrom_label(prev_chrom, chrom_start, xi))
            chrom_start = xi
            prev_chrom = chrom
    # handle final chromosome
    if prev_chrom is not None:
        xtick_pos.append((chrom_start + len(arm_order) - 1) / 2)
        xtick_lab.append(_chrom_label(prev_chrom, chrom_start, len(arm_order)))

    ax.set_xticks(xtick_pos)
    ax.set_xticklabels(xtick_lab, rotation=0)
    ax.set_xlim(-0.5, len(arm_order) - 0.5)
    ax.xaxis.set_tick_params(length=0)

    ax.set_yticks(range(len(row_labels)))
    ax.set_yticklabels(row_labels)

    for i in range(2, len(row_labels), 2):
        ax.axhline(i - 0.5, color="white", linewidth=2.0, zorder=3)
    for i in range(1, len(row_labels), 2):
        ax.axhline(i - 0.5, color="white", linewidth=0.6, zorder=3)

    ax.set_xlabel("Chromosome arm")

    sm = ScalarMappable(cmap=cmap, norm=plot_norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.02, pad=0.01)
    cbar.set_label(r"log$_2$ scale value")
    cbar.ax.tick_params(labelsize=FONT_SIZE - 2)

    return fig, ax


fig, ax = plot_combined_arm_heatmap(arm_level_by_trio, FUSION_TRIOS)

save_fig(fig, "arm_level_dosage_heatmap")
plt.show()

## 9. Gene-First: Top DE Genes and Their CNV State

Section 4's correlation and Section 5's boxplot both ask whether dosage
matters *on average, across all genes*. This section instead starts from
the individual genes most likely to be named in the paper:
the most confidently, most strongly differentially expressed genes per
fusion trio, and asks what CNV state each one is sitting in.

Genes are first filtered to `f_mpv_padj < PADJ_THRESHOLD_TOP_GENES`
(significance gate), then ranked by `|f_mpv_log2FoldChange|` and the top
`TOP_N_GENES` taken per trio. All genes in `gene_level_data` already sit
on a CNV segment by construction (Section 3 only keeps genes that
mapped to one), so no separate CNV-availability filter is needed.

For each top gene, the segment's `classification` (under/match/over)
predicts a direction: over-duplicated segments predict `log2FC > 0`,
under-duplicated predict `log2FC < 0`. Genes on `match` segments have no
directional prediction and are reported but excluded from the
concordance rate. Per trio, this reports the fraction of directional
top genes whose observed direction matches the CNV-predicted direction,
tested against a 50% chance rate with a one-sided binomial test, then
FDR-corrected (Benjamini-Hochberg) across trios.

In [ ]:
# ============================================================
# SECTION 9 CONFIGURATION - Gene-first: top DE genes vs CNV state
# ============================================================
TOP_N_GENES = 25                  # adjustable: how many top genes per trio to report
PADJ_THRESHOLD_TOP_GENES = 0.05   # significance gate applied before ranking by |log2FC|


def benjamini_hochberg(pvals):
    """
    Standard Benjamini-Hochberg FDR correction, implemented locally so
    this notebook does not pick up a statsmodels dependency just for
    this. NaNs are ignored (returned as NaN) and do not participate in
    the ranking of the real p-values.
    """
    pvals = np.asarray(pvals, dtype=float)
    out = np.full(pvals.shape, np.nan)
    valid = ~np.isnan(pvals)
    if valid.sum() == 0:
        return out

    p_valid = pvals[valid]
    n = len(p_valid)
    order = np.argsort(p_valid)
    ranked = p_valid[order]
    q = ranked * n / np.arange(1, n + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0, 1)

    q_out = np.empty(n)
    q_out[order] = q
    out[valid] = q_out
    return out


def get_top_de_genes(df, top_n, padj_threshold,
                      padj_col="f_mpv_padj", lfc_col="f_mpv_log2FoldChange"):
    """
    Filters to significant genes (padj < padj_threshold), ranks by
    |log2FC| descending, and returns the top_n. Adds expected_direction
    (from the CNV classification the gene's segment already carries),
    observed_direction (sign of log2FC), and concordant (whether they
    match; NaN for genes on 'match' segments, which have no directional
    prediction).
    """
    sub = df.dropna(subset=[padj_col, lfc_col]).copy()
    sig = sub.loc[sub[padj_col] < padj_threshold].copy()
    sig["abs_log2FC"] = sig[lfc_col].abs()
    top = sig.sort_values("abs_log2FC", ascending=False).head(top_n).copy()

    direction_map = {"over": "up", "under": "down", "match": None}
    top["expected_direction"] = top["classification"].map(direction_map)
    top["observed_direction"] = np.where(top[lfc_col] > 0, "up", "down")
    top["concordant"] = np.where(
        top["expected_direction"].isna(),
        np.nan,
        top["expected_direction"] == top["observed_direction"],
    )
    return top.sort_values("abs_log2FC", ascending=False)

In [ ]:
top_gene_tables = {}

for trio_name in FUSION_TRIOS.keys():
    df = gene_level_data[trio_name]
    top = get_top_de_genes(df, TOP_N_GENES, PADJ_THRESHOLD_TOP_GENES)
    top_gene_tables[trio_name] = top

    out_path = os.path.join(OUTPUT_DIR, f"{trio_name}_top_{TOP_N_GENES}_de_genes_vs_cnv.tsv")
    top.to_csv(out_path, sep="\t")

    print(f"{trio_name}: {len(top)} top genes "
          f"(padj < {PADJ_THRESHOLD_TOP_GENES}, ranked by |log2FC|) -> {out_path}")

display(top_gene_tables[list(FUSION_TRIOS.keys())[0]].head(10))

In [ ]:
top_gene_summary_rows = []

for trio_name in FUSION_TRIOS.keys():
    t = top_gene_tables[trio_name]
    directional = t.dropna(subset=["expected_direction"])
    n_dir = len(directional)
    n_conc = int(directional["concordant"].sum()) if n_dir else 0
    rate = n_conc / n_dir if n_dir else np.nan
    p = binomtest(n_conc, n_dir, p=0.5, alternative="greater").pvalue if n_dir else np.nan

    top_gene_summary_rows.append({
        "trio": trio_name,
        "n_top_genes": len(t),
        "n_over": int((t["classification"] == "over").sum()),
        "n_under": int((t["classification"] == "under").sum()),
        "n_match": int((t["classification"] == "match").sum()),
        "n_directional": n_dir,
        "n_concordant": n_conc,
        "pct_concordant": rate,
        "binom_p_greater_than_chance": p,
    })

top_gene_summary_df = pd.DataFrame(top_gene_summary_rows)
valid = top_gene_summary_df["binom_p_greater_than_chance"].notna()
top_gene_summary_df.loc[valid, "binom_p_fdr_bh"] = benjamini_hochberg(
    top_gene_summary_df.loc[valid, "binom_p_greater_than_chance"].values
)

out_path = os.path.join(OUTPUT_DIR, f"top_{TOP_N_GENES}_de_genes_concordance_summary.tsv")
top_gene_summary_df.to_csv(out_path, sep="\t", index=False)

print(f"Top-{TOP_N_GENES} DE gene concordance summary (one sided binomial vs 50% chance, "
      f"BH-FDR corrected across {len(top_gene_summary_df)} trios):")
display(top_gene_summary_df)

In [ ]:
def plot_labeled_top_genes(gene_level_df, top_genes_df, trio_name, ax,
                            x_col="log2_residual", y_col="f_mpv_log2FoldChange"):
    """
    Draws the same scatter as plot_dosage_correlation (Section 4), then
    overlays open circles and gene-symbol labels for the top DE gene set
    so specific candidates can be picked out visually.
    """
    stats_dict = plot_dosage_correlation(gene_level_df, trio_name, ax, x_col=x_col, y_col=y_col)

    for gene, row in top_genes_df.iterrows():
        x, y = row[x_col], row[y_col]
        if pd.isna(x) or pd.isna(y):
            continue
        ax.scatter([x], [y], s=20, facecolors="none", edgecolors="black", linewidths=0.9, zorder=5)
        ax.annotate(gene, (x, y), fontsize=FONT_SIZE - 6, color="black",
                    xytext=(3, 3), textcoords="offset points", zorder=6)

    return stats_dict


fig, axes = plt.subplots(1, len(FUSION_TRIOS), figsize=(6 * len(FUSION_TRIOS), 6), constrained_layout=True)
if len(FUSION_TRIOS) == 1:
    axes = [axes]

for ax, trio_name in zip(axes, FUSION_TRIOS.keys()):
    plot_labeled_top_genes(gene_level_data[trio_name], top_gene_tables[trio_name], trio_name, ax)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=len(labels), frameon=False,
           bbox_to_anchor=(0.5, -0.18), markerscale=3)

save_fig(fig, f"top_{TOP_N_GENES}_de_genes_labeled_scatter")
plt.show()

## 10. All Significant DE Genes: Dosage-Explained vs. Not

Section 9 starts from a curated subset (the top DE genes per trio). This
section instead takes every gene meeting a DE significance + effect-size bar
and asks a more basic question: of all the real DE signal, what fraction
sits on a locus with real copy-number movement at all, versus a copy-neutral
locus where CNV dosage cannot be the explanation? Genes in the copy-neutral
bucket are, by construction, NOT dosage-explainable; whatever expression
change they show has to come from something else (trans effects,
fusion-specific regulatory rewiring, etc.), so the size of that bucket is a
direct estimate of how much of the DE signal dosage does *not* explain.

Both an arm-level and a segment-level version of the "real CN movement"
threshold are computed. Depends on functions and variables defined earlier
in this section (`compute_arm_level_residual_raw`) and earlier in the
notebook (`cnv_data` / `gene_level_data`, Sections 1 and 3) - run those
cells first if this is a fresh kernel.

Produces two tables:

- **Table A**: per trio, the headline fraction of significant DE genes
  on a high-|residual| locus vs. a neutral one (arm-level and
  segment-level side by side).
- **Table B (pooled)**: the DESeq2 category breakdown (conserved,
  fusion-altered, transgressive_up, etc.) across all significant DE
  genes, regardless of CN state - the overall shape of the DE signal.


In [ ]:
# ============================================================
# SECTION 11 CONFIGURATION - All significant DE genes: dosage-explained vs not
# ============================================================
# PADJ_THRESH_DE is redefined here (identical value to the Section 0
# config cell) so this section is self-contained and can be copied or
# rerun on its own without depending on execution order elsewhere.
PADJ_THRESH_DE = 0.05
DE_ABS_LOG2FC_THRESHOLD = 0.585        # significant DE gene must also clear this |log2FC| (1.0 = 2-fold)
DOSAGE_RESIDUAL_THRESHOLD = 1.0      # |residual| (linear copy-number units) cutoff for "real CN movement"

In [ ]:
def get_significant_de_genes(df, padj_thresh, abs_lfc_thresh,
                              padj_col="f_mpv_padj", lfc_col="f_mpv_log2FoldChange"):
    """
    All genes clearing both a significance and an effect-size bar;
    the full DE gene pool this section works from, not a top-N subset.
    """
    sub = df.dropna(subset=[padj_col, lfc_col])
    return sub.loc[(sub[padj_col] < padj_thresh) & (sub[lfc_col].abs() >= abs_lfc_thresh)].copy()

def compute_arm_level_residual_raw(seg_df, residual_col="residual"):
    """
    Same bp-weighted-mean convention as compute_arm_level_cnv (Section
    6), applied to the raw (linear, copy-number-unit) residual column
    instead of log2_residual, so extreme arms can be selected in
    absolute copy-number-deviation terms.
    """
    rows = []
    for arm, g in seg_df.groupby("Arm"):
        g_valid = g.dropna(subset=[residual_col])
        total_bp = g_valid["segment_bp"].sum() if "segment_bp" in g_valid.columns else 0
        weighted_mean = (
            np.average(g_valid[residual_col], weights=g_valid["segment_bp"])
            if len(g_valid) and total_bp else np.nan
        )
        rows.append({"Arm": arm, "arm_residual": weighted_mean, "n_segments": len(g)})
    return pd.DataFrame(rows).set_index("Arm")

def add_arm_residual_bucket(sig_df, arm_residual_lookup, threshold):
    """
    Attaches each gene's arm-level bp-weighted residual (computed by
    compute_arm_level_residual_raw above) and buckets it:
    high_residual (|arm_residual| >= threshold, real CN movement at the
    arm level), neutral (below threshold), or no_data (arm missing from
    the CNV segment table entirely).
    """
    out = sig_df.join(arm_residual_lookup[["arm_residual"]], on="Arm")
    conditions = [out["arm_residual"].isna(), out["arm_residual"].abs() >= threshold]
    out["arm_dosage_bucket"] = np.select(conditions, ["no_data", "high_residual"], default="neutral")
    return out


def add_segment_residual_bucket(sig_df, threshold):
    """
    Buckets each gene on its own segment's raw residual, already
    attached at the gene level in gene_level_data as 'residual'; no
    extra lookup needed, since each gene's segment assignment is fixed
    at the Section 3 mapping step.
    """
    out = sig_df.copy()
    conditions = [out["residual"].isna(), out["residual"].abs() >= threshold]
    out["segment_dosage_bucket"] = np.select(conditions, ["no_data", "high_residual"], default="neutral")
    return out


In [ ]:
sig_de_gene_tables = {}

for trio_name in FUSION_TRIOS.keys():
    df = gene_level_data[trio_name]
    sig = get_significant_de_genes(df, PADJ_THRESH_DE, DE_ABS_LOG2FC_THRESHOLD)

    arm_residual_lookup = compute_arm_level_residual_raw(cnv_data[trio_name])
    sig = add_arm_residual_bucket(sig, arm_residual_lookup, DOSAGE_RESIDUAL_THRESHOLD)
    sig = add_segment_residual_bucket(sig, DOSAGE_RESIDUAL_THRESHOLD)

    sig_de_gene_tables[trio_name] = sig

    out_path = os.path.join(OUTPUT_DIR, f"{trio_name}_significant_DE_genes_dosage_buckets.tsv")
    sig.to_csv(out_path, sep="\t")

    print(f"{trio_name}: {len(sig)} genes pass padj < {PADJ_THRESH_DE} and "
          f"|log2FC| >= {DE_ABS_LOG2FC_THRESHOLD} -> {out_path}")

In [ ]:
# Diagnostic: which significant DE genes (by the tagseq-side criteria alone,
# before any CNV join) are missing from gene_level_data, and are they
# concentrated on a particular chromosome (e.g. Y) or scattered?
for trio_name in FUSION_TRIOS.keys():
    de_df = deseq_data[trio_name]  # raw DESeq2 table, pre-CNV-join
    sig_de_only = de_df.loc[
        (de_df["f_mpv_padj"] < PADJ_THRESH_DE)
        & (de_df["f_mpv_log2FoldChange"].abs() >= DE_ABS_LOG2FC_THRESHOLD)
    ]
    missing = sig_de_only.index.difference(gene_level_data[trio_name].index)

    print(f"{trio_name}: {len(sig_de_only)} significant in raw DESeq2 table, "
          f"{len(missing)} missing from gene_level_data")
    if len(missing):
        missing_coords = gene_coords.reindex(missing)
        print("  Missing genes by chrom (NaN chrom = failed mygene lookup entirely):")
        print(missing_coords["chrom"].value_counts(dropna=False).to_string())
        print("  Sample missing gene symbols:", list(missing[:15]))
    print()

In [ ]:
table_a_rows = []

for trio_name in FUSION_TRIOS.keys():
    sig = sig_de_gene_tables[trio_name]
    n_total = len(sig)

    for level, bucket_col in [("arm", "arm_dosage_bucket"), ("segment", "segment_dosage_bucket")]:
        counts = sig[bucket_col].value_counts()
        n_high = int(counts.get("high_residual", 0))
        n_neutral = int(counts.get("neutral", 0))
        n_no_data = int(counts.get("no_data", 0))
        n_scored = n_high + n_neutral

        table_a_rows.append({
            "trio": trio_name,
            "level": level,
            "n_significant_DE_genes": n_total,
            "n_high_residual": n_high,
            "n_neutral": n_neutral,
            "n_no_cnv_data": n_no_data,
            # denominator excludes no_data genes -- of genes we could actually score
            "pct_high_residual_of_scored": n_high / n_scored if n_scored else np.nan,
            # denominator is every significant DE gene -- the headline "fraction dosage-explainable" number
            "pct_high_residual_of_all_sig": n_high / n_total if n_total else np.nan,
        })

table_a_df = pd.DataFrame(table_a_rows)
out_path = os.path.join(OUTPUT_DIR, "dosage_explained_fraction_summary.tsv")
table_a_df.to_csv(out_path, sep="\t", index=False)

print("Table A - fraction of significant DE genes on a locus with real CN movement "
      f"(|residual| >= {DOSAGE_RESIDUAL_THRESHOLD} copy-number units):")
display(table_a_df)

In [ ]:
def enrichment_vs_background(sig_df, full_df, bucket_col_sig, residual_lookup=None,
                              is_segment_level=False, threshold=DOSAGE_RESIDUAL_THRESHOLD):
    """
    Fisher's exact test: are significant DE genes enriched on high-residual
    loci relative to the same locus-level background rate among ALL
    genes in gene_level_data (significant or not)? no_data genes are
    excluded from both groups, since they have no CN state to compare.
    """
    if is_segment_level:
        bg = full_df.copy()
        conditions = [bg["residual"].isna(), bg["residual"].abs() >= threshold]
        bg["bucket"] = np.select(conditions, ["no_data", "high_residual"], default="neutral")
    else:
        bg = full_df.join(residual_lookup[["arm_residual"]], on="Arm")
        conditions = [bg["arm_residual"].isna(), bg["arm_residual"].abs() >= threshold]
        bg["bucket"] = np.select(conditions, ["no_data", "high_residual"], default="neutral")

    bg_scored = bg.loc[bg["bucket"].isin(["high_residual", "neutral"])]
    n_bg = len(bg_scored)
    k_bg = int((bg_scored["bucket"] == "high_residual").sum())

    sig_scored = sig_df.loc[sig_df[bucket_col_sig].isin(["high_residual", "neutral"])]
    n_sig = len(sig_scored)
    k_sig = int((sig_scored[bucket_col_sig] == "high_residual").sum())

    if n_sig == 0 or n_bg == 0:
        return {"n_sig_scored": n_sig, "n_sig_high_residual": k_sig, "pct_sig_high_residual": np.nan,
                "n_background_scored": n_bg, "n_background_high_residual": k_bg,
                "pct_background_high_residual": np.nan, "fisher_odds_ratio": np.nan, "fisher_p": np.nan}

    table = [[k_sig, n_sig - k_sig], [k_bg, n_bg - k_bg]]
    odds_ratio, p = fisher_exact(table)

    return {
        "n_sig_scored": n_sig, "n_sig_high_residual": k_sig,
        "pct_sig_high_residual": k_sig / n_sig,
        "n_background_scored": n_bg, "n_background_high_residual": k_bg,
        "pct_background_high_residual": k_bg / n_bg,
        "fisher_odds_ratio": odds_ratio, "fisher_p": p,
    }


table_a_enrichment_rows = []

for trio_name in FUSION_TRIOS.keys():
    sig = sig_de_gene_tables[trio_name]
    full = gene_level_data[trio_name]

    arm_residual_lookup = compute_arm_level_residual_raw(cnv_data[trio_name])
    for level, bucket_col, is_seg in [("arm", "arm_dosage_bucket", False), ("segment", "segment_dosage_bucket", True)]:
        result = enrichment_vs_background(
            sig, full, bucket_col,
            residual_lookup=arm_residual_lookup if not is_seg else None,
            is_segment_level=is_seg,
        )
        result["trio"] = trio_name
        result["level"] = level
        table_a_enrichment_rows.append(result)

table_a_enrichment_df = pd.DataFrame(table_a_enrichment_rows)
cols = ["trio", "level", "n_sig_scored", "n_sig_high_residual", "pct_sig_high_residual",
        "n_background_scored", "n_background_high_residual", "pct_background_high_residual",
        "fisher_odds_ratio", "fisher_p"]
table_a_enrichment_df = table_a_enrichment_df[cols]

valid = table_a_enrichment_df["fisher_p"].notna()
table_a_enrichment_df.loc[valid, "fisher_p_fdr_bh"] = benjamini_hochberg(
    table_a_enrichment_df.loc[valid, "fisher_p"].values
)

out_path = os.path.join(OUTPUT_DIR, "dosage_explained_enrichment_vs_background.tsv")
table_a_enrichment_df.to_csv(out_path, sep="\t", index=False)

print("Table A enrichment - significant DE genes' high-residual rate vs. background rate "
      "(all genes in gene_level_data), Fisher's exact test:")
display(table_a_enrichment_df)

In [ ]:
def plot_dosage_bucket_barplot(sig_de_gene_tables, fusion_trios, mode="both",
                                bar_width=None, bar_gap=0.04, group_spacing=1.0,
                                figsize=None, legend_gap=12, bottom_margin=None):
    """
    One 100% stacked bar per fusion trio per level, colored by
    high_residual vs neutral vs no_data, restricted to genes meeting
    the DE significance/effect-size cutoffs already applied when
    sig_de_gene_tables was built.

    mode controls which level(s) are drawn:
      "arm"     - one bar per trio, arm-level bucket only
      "segment" - one bar per trio, segment-level bucket only
      "both"    - two bars per trio (arm, segment), grouped together

    bar_width controls the width of each individual bar; None keeps the
    previous defaults (0.34 for mode="both", 0.6 otherwise). bar_gap is
    the gap between the two bars within a trio group (mode="both" only).
    group_spacing scales the distance between trio group centers (1.0
    matches the original spacing).

    figsize sets the figure size directly. If None, the width is
    computed automatically from bar_width/bar_gap/group_spacing so it
    shrinks or grows along with the bars instead of staying fixed.

    All x-axis labeling is drawn manually (default tick labels are
    turned off) so the per-bar level label ("arm"/"segment", only shown
    in "both" mode) and the per-group trio label never collide, however
    many levels are being shown.
    """
    if mode not in ("arm", "segment", "both"):
        raise ValueError(f"Unknown mode: {mode!r}, must be 'arm', 'segment', or 'both'")

    level_defs = {"arm": ("arm", "arm_dosage_bucket"), "segment": ("segment", "segment_dosage_bucket")}
    levels = [level_defs["arm"], level_defs["segment"]] if mode == "both" else [level_defs[mode]]
    n_levels = len(levels)

    trio_names = list(fusion_trios.keys())
    x = np.arange(len(trio_names)) * group_spacing

    if bar_width is None:
        bar_width = 0.34 if n_levels > 1 else 0.6
    step = bar_width + bar_gap if n_levels > 1 else bar_width

    BUCKET_COLORS = {
        "high_residual": CLASS_COLORS["over"],
        "neutral": "#D9D9D9",
        "no_data": "#F3F4F6",
    }
    BUCKET_LABELS = {
        "high_residual": "High |residual|\n(CNV altered)",
        "neutral": "Neutral\n(no CN change)",
        "no_data": "No CNV data",
    }

    if figsize is None:
        # cluster_width is how much horizontal space one trio's bar(s)
        # actually occupy; figure width scales with that plus the
        # spacing between trio groups, so shrinking bar_width or
        # group_spacing shrinks the figure instead of leaving empty space.
        cluster_width = bar_width if n_levels == 1 else bar_width * n_levels + bar_gap * (n_levels - 1)
        fig_width = len(trio_names) * (cluster_width + group_spacing) + 1.5
        figsize = (fig_width, 5.5)

    fig, ax = plt.subplots(figsize=figsize)

    any_no_data = False
    bar_centers = {t: [] for t in trio_names}

    for li, (level, bucket_col) in enumerate(levels):
        offset = x if n_levels == 1 else x + (li - (n_levels - 1) / 2) * step
        bottoms = np.zeros(len(trio_names))

        pct_by_trio = {}
        for trio_name in trio_names:
            sig = sig_de_gene_tables[trio_name]
            counts = sig[bucket_col].value_counts()
            total = len(sig)
            pct_by_trio[trio_name] = {
                b: (counts.get(b, 0) / total * 100 if total else 0.0)
                for b in ["high_residual", "neutral", "no_data"]
            }
            if counts.get("no_data", 0) > 0:
                any_no_data = True

        for bucket in ["neutral", "high_residual", "no_data"]:
            vals = np.array([pct_by_trio[t][bucket] for t in trio_names])
            ax.bar(offset, vals, bar_width, bottom=bottoms, color=BUCKET_COLORS[bucket],
                   edgecolor="white", linewidth=0.6,
                   label=BUCKET_LABELS[bucket] if li == 0 else None)
            bottoms += vals

        for xi, trio_name in zip(offset, trio_names):
            bar_centers[trio_name].append(xi)
            if n_levels > 1:
                # level label, close under its own bar
                ax.text(xi, -2.5, level, ha="center", va="top", fontsize=FONT_SIZE - 2.5,
                         color="#666666", clip_on=False)

    ax.set_xticks([])
    ax.set_xlim(x[0] - 0.6 * group_spacing, x[-1] + 0.6 * group_spacing)
    ax.set_ylim(0, 100)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_ylabel("% of significant DE genes")

    # trio label, centered under the group of bars, well below the level labels
    trio_label_y = -11 if n_levels > 1 else -4
    for trio_name in trio_names:
        xc = float(np.mean(bar_centers[trio_name]))
        ax.text(xc, trio_label_y, f"Fusion\n{trio_name}", ha="center", va="top",
                 fontsize=FONT_SIZE, clip_on=False)

    handles, labels = ax.get_legend_handles_labels()
    if not any_no_data:
        keep = [i for i, l in enumerate(labels) if l != BUCKET_LABELS["no_data"]]
        handles, labels = [handles[i] for i in keep], [labels[i] for i in keep]

    # legend sits legend_gap below the trio labels, in the same units as
    # the y-axis (0-100), then converted to the 0-1 axes-fraction that
    # bbox_to_anchor expects. Increase legend_gap for more breathing room.
    legend_y = (trio_label_y - legend_gap) / 100
    leg=ax.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, legend_y),
              ncol=len(labels), frameon=False, fontsize=FONT_SIZE-2)

    for text in leg.get_texts():
        text.set_multialignment('center')

    # bottom margin must reserve enough figure height for level labels +
    # trio labels + legend_gap + the legend itself, or the legend gets
    # clipped. Grows automatically as legend_y moves further down unless
    # you pass bottom_margin explicitly.
    if bottom_margin is None:
        bottom_margin = min(0.55, 0.22 + abs(legend_y) * 0.6)
    fig.subplots_adjust(bottom=bottom_margin)

    return fig, ax

DOSAGE_BARPLOT_MODE = "segment"  # "arm", "segment", or "both"

fig, ax = plot_dosage_bucket_barplot(
    sig_de_gene_tables, FUSION_TRIOS, mode=DOSAGE_BARPLOT_MODE,
    bar_width=0.6, bar_gap=0.06, group_spacing=.8, figsize=(3.5,5.5),
)
save_fig(fig, f"dosage_bucket_barplot_by_clone_{DOSAGE_BARPLOT_MODE}")
plt.show()

In [ ]:
table_b_pooled_rows = []

CATEGORY_ORDER = [
    "conserved", "fusion-altered", "additive",
    "dominant_P1", "dominant_P2",
    "transgressive_up", "transgressive_down", "ambiguous_nonadditive",
]

for trio_name in FUSION_TRIOS.keys():
    sig = sig_de_gene_tables[trio_name]
    n_total = len(sig)
    cat_counts = sig["category"].value_counts()

    for category in CATEGORY_ORDER:
        n_cat = int(cat_counts.get(category, 0))
        table_b_pooled_rows.append({
            "trio": trio_name, "category": category,
            "n_genes": n_cat, "pct_of_all_sig_DE_genes": n_cat / n_total if n_total else np.nan,
        })

table_b_pooled_df = pd.DataFrame(table_b_pooled_rows)
out_path = os.path.join(OUTPUT_DIR, "dosage_explained_category_breakdown_pooled.tsv")
table_b_pooled_df.to_csv(out_path, sep="\t", index=False)

print("Table B (pooled) - category breakdown across all significant DE genes, regardless of CN state:")
display(table_b_pooled_df.pivot(index="trio", columns="category", values="pct_of_all_sig_DE_genes").round(3))